# Hunyuan3D Testgrounds

## Import the package from other folder

In [ ]:
from context import drm, inference

In [ ]:
import os
import glob
from inference import Hunyuan3DOmniSiTFlowMatchingPipeline, infer_bbox,infer_pose, infer_point, infer_voxel

gpu_id = 0
num_gpu = 1
save_dir = "./omni_inference_results"
control_type = "point" #"voxel", "point", 'pose', 'bbox'
repo_id = "tencent/Hunyuan3D-Omni"
use_ema = True
flashvdm = True



# Demonstration of various-condition 3D generation capabilities
print("=" * 80)
print("HUNYUAN3D-OMNI VARIOUS-CONDITION INFERENCE DEMO")
print("=" * 80)

# initial
print(f"From Pretrained: {repo_id}")
pipeline = Hunyuan3DOmniSiTFlowMatchingPipeline.from_pretrained(
    repo_id, 
    fast_decode=flashvdm
)

# 1. Bounding Box Control Inference
if control_type == "bbox":
    print("\n" + "=" * 80)
    print("1. Running Bounding Box Control Inference...")
    bbox_data_path = "./demos/bbox/data.json"
    bbox_output_dir = os.path.join(save_dir, "3domni_bbox")
    infer_bbox(pipeline, bbox_data_path, bbox_output_dir)
    print("Finished Bounding Box Control Inference")

# 2. Pose Control Inference
if control_type == "pose":
    print("\n" + "=" * 80)
    print("2. Running Pose Control Inference...")
    pose_configs = {
        "a_pose": "./demos/pose/a_pose_bone.txt",
        "handup_pose": "./demos/pose/handup_pose_bone.txt",
        "sky_pose": "./demos/pose/sky_pose_bone.txt",
    }
    pose_images = glob.glob("./demos/pose/*.png")
    pose_output_dir = os.path.join(save_dir, "3domni_pose")
    infer_pose(pipeline, pose_images, pose_configs, pose_output_dir)
    print("Finished Pose Control Inference")

# 3. Point Cloud Control Inference
if control_type == "point":
    print("\n" + "=" * 80)
    print("3. Running Point Cloud Control Inference...")
    point_data_path = "./demos/point/data.json"
    point_output_dir = os.path.join(save_dir, "3domni_point")
    infer_point(pipeline,  point_data_path, point_output_dir)
    print("Finished Point Cloud Control Inference")

# 4. Voxel Control Inference
if control_type == "voxel":
    print("\n" + "=" * 80)
    print("4. Running Voxel Control Inference...")
    voxel_data_path = "./demos/voxel/data.json"
    voxel_output_dir = os.path.join(save_dir, "3domni_voxel")
    infer_voxel(pipeline, voxel_data_path, voxel_output_dir)
    print("Finished Voxel Control Inference")

print("\n" + "=" * 80)
print("INFERENCE COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
import os
import glob
from inference import Hunyuan3DOmniSiTFlowMatchingPipeline, infer_bbox,infer_pose, infer_point, infer_voxel,FloaterRemover, DegenerateFaceRemover

gpu_id = 0
num_gpu = 1
save_dir = "./omni_inference_results"
control_type = "point" #"voxel", "point", 'pose', 'bbox'
repo_id = "tencent/Hunyuan3D-Omni"
use_ema = True
flashvdm = True

point_data_path = "./demos/point/data.json"
point_output_dir = os.path.join(save_dir, "3domni_point")

# Load the pipeline
pipeline = Hunyuan3DOmniSiTFlowMatchingPipeline.from_pretrained(
    repo_id, 
    fast_decode=flashvdm
)

# Infer the resulting mesh from the input point
infer_point(pipeline,  point_data_path, point_output_dir)
print("Finished Point Cloud Control Inference")



## Single Point/image demo

In [ ]:
import trimesh
import torch
from context import inference
from inference import Hunyuan3DOmniSiTFlowMatchingPipeline,FloaterRemover, DegenerateFaceRemover


pointPath = ""
imagePath = ""

os.makedirs(save_dir, exist_ok=True)

# Load the pipeline
pipeline = Hunyuan3DOmniSiTFlowMatchingPipeline.from_pretrained(
    "tencent/Hunyuan3D-Omni", 
    fast_decode = True
)

mesh = trimesh.load(pointPath)
mesh = inference.normalize_mesh(mesh, scale=0.98)
surface = mesh.vertices
surface = torch.FloatTensor(surface).unsqueeze(0)
surface = surface.to(pipeline.device).to(pipeline.dtype)
result = pipeline(
    image=imagePath,
    point=surface,
    num_inference_steps=20,
    octree_resolution=64,
    mc_level=0,
    guidance_scale=4.5,
    generator=torch.Generator('cuda').manual_seed(1234),
)

mesh = result['shapes'][0][0]#[0]
sampled_point = result['sampled_point'][0]#[0]
mesh = FloaterRemover()(mesh)
mesh = DegenerateFaceRemover()(mesh)

mesh.show()